In [1]:

from pathlib import Path
import pandas as pd

# Folder containing the original LinkedIn datasets
DATA_DIR = Path(r"E:\Analytic Plan BI\LinkedIn Dataset")

# File paths
JOBS_FILE = DATA_DIR / "linkedin_job_postings.csv"
SKILLS_FILE = DATA_DIR / "job_skills.csv"

# Check whether both files exist
for file in [JOBS_FILE, SKILLS_FILE]:
    print(f"File: {file.name}")
    print(f"Exists: {file.exists()}")

    if file.exists():
        print(f"Size: {file.stat().st_size / (1024**2):,.2f} MB")

    print("-" * 50)

# Read only five rows to avoid memory issues
if JOBS_FILE.exists() and SKILLS_FILE.exists():

    jobs_sample = pd.read_csv(JOBS_FILE, nrows=5)
    skills_sample = pd.read_csv(SKILLS_FILE, nrows=5)

    print("\nJOB POSTINGS COLUMNS:")
    print(jobs_sample.columns.tolist())

    print("\nJOB SKILLS COLUMNS:")
    print(skills_sample.columns.tolist())

    print("\nJOB POSTINGS SAMPLE:")
    display(jobs_sample.head())

    print("\nJOB SKILLS SAMPLE:")
    display(skills_sample.head())

else:
    print("Please check the folder path or filenames.")


File: linkedin_job_postings.csv
Exists: True
Size: 396.09 MB
--------------------------------------------------
File: job_skills.csv
Exists: True
Size: 641.55 MB
--------------------------------------------------

JOB POSTINGS COLUMNS:
['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type']

JOB SKILLS COLUMNS:
['job_link', 'job_skills']

JOB POSTINGS SAMPLE:


,job_link,last_processed_time,got_summary,got_ner,is_being_worked,job_title,company,job_location,first_seen,search_city,search_country,search_position,job_level,job_type
0,https://www.linkedin.com/jobs/view/account-exe...,2024-01-21 07:12:29.00256+00,t,t,f,Account Executive - Dispensing (NorCal/Norther...,BD,"San Diego, CA",2024-01-15,Coronado,United States,Color Maker,Mid senior,Onsite
1,https://www.linkedin.com/jobs/view/registered-...,2024-01-21 07:39:58.88137+00,t,t,f,Registered Nurse - RN Care Manager,Trinity Health MI,"Norton Shores, MI",2024-01-14,Grand Haven,United States,Director Nursing Service,Mid senior,Onsite
2,https://www.linkedin.com/jobs/view/restaurant-...,2024-01-21 07:40:00.251126+00,t,t,f,RESTAURANT SUPERVISOR - THE FORKLIFT,Wasatch Adaptive Sports,"Sandy, UT",2024-01-14,Tooele,United States,Stand-In,Mid senior,Onsite
3,https://www.linkedin.com/jobs/view/independent...,2024-01-21 07:40:00.308133+00,t,t,f,Independent Real Estate Agent,Howard Hanna | Rand Realty,"Englewood Cliffs, NJ",2024-01-16,Pinehurst,United States,Real-Estate Clerk,Mid senior,Onsite
4,https://www.linkedin.com/jobs/view/group-unit-...,2024-01-19 09:45:09.215838+00,f,f,f,Group/Unit Supervisor (Systems Support Manager...,"IRS, Office of Chief Counsel","Chamblee, GA",2024-01-17,Gadsden,United States,Supervisor Travel-Information Center,Mid senior,Onsite



JOB SKILLS SAMPLE:


,job_link,job_skills
0,https://www.linkedin.com/jobs/view/housekeeper...,"Building Custodial Services, Cleaning, Janitor..."
1,https://www.linkedin.com/jobs/view/assistant-g...,"Customer service, Restaurant management, Food ..."
2,https://www.linkedin.com/jobs/view/school-base...,"Applied Behavior Analysis (ABA), Data analysis..."
3,https://www.linkedin.com/jobs/view/electrical-...,"Electrical Engineering, Project Controls, Sche..."
4,https://www.linkedin.com/jobs/view/electrical-...,"Electrical Assembly, Point to point wiring, St..."


In [2]:

# Create a dedicated Assignment 2 output folder

OUTPUT_DIR = DATA_DIR / "Assignment_2_Analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTPUT_DIR)
print("Folder exists:", OUTPUT_DIR.exists())


Output folder: E:\Analytic Plan BI\LinkedIn Dataset\Assignment_2_Analysis
Folder exists: True


In [3]:

import pandas as pd

# Columns needed for the analysis
required_columns = [
    "job_link",
    "job_title",
    "company",
    "job_location",
    "first_seen",
    "search_city",
    "search_country",
    "search_position",
    "job_level",
    "job_type"
]

# Strict title matching, consistent with Assignment 1
ba_pattern = r"\bbusiness analyst\b"

ba_chunks = []

total_postings = 0
total_ba_postings = 0
total_us_ba_search = 0

for chunk in pd.read_csv(
    JOBS_FILE,
    usecols=required_columns,
    chunksize=50000,
    dtype=str,
    low_memory=False
):

    total_postings += len(chunk)

    # Identify titles containing the exact phrase
    ba_mask = chunk["job_title"].str.contains(
        ba_pattern,
        case=False,
        regex=True,
        na=False
    )

    ba_chunk = chunk.loc[ba_mask].copy()

    total_ba_postings += len(ba_chunk)

    # US-search subset
    us_mask = (
        ba_chunk["search_country"]
        .fillna("")
        .str.strip()
        .str.casefold()
        .eq("united states")
    )

    total_us_ba_search += us_mask.sum()

    if not ba_chunk.empty:
        ba_chunks.append(ba_chunk)

# Combine the small BA subsets
all_ba = pd.concat(
    ba_chunks,
    ignore_index=True
)

# Extract US-search BA advertisements
us_ba_search = all_ba[
    all_ba["search_country"]
    .fillna("")
    .str.strip()
    .str.casefold()
    .eq("united states")
].copy()

# Save intermediate outputs
all_ba.to_csv(
    OUTPUT_DIR / "ALL_BA_POSTINGS.csv",
    index=False
)

us_ba_search.to_csv(
    OUTPUT_DIR / "US_BA_SEARCH_POSTINGS.csv",
    index=False
)

# Verification summary
print("=" * 55)
print("ASSIGNMENT 2 - BA POPULATION VERIFICATION")
print("=" * 55)

print(f"Total job postings: {total_postings:,}")
print(f"All explicit BA postings: {total_ba_postings:,}")
print(f"US-search BA postings: {total_us_ba_search:,}")

print("\nExpected Assignment 1 counts:")
print("All explicit BA postings: 4,546")
print("US-search BA postings: 3,732")

print("\nUnique job links:")
print("All BA:", all_ba["job_link"].nunique())
print("US-search BA:", us_ba_search["job_link"].nunique())

print("\nSaved files:")
print("ALL_BA_POSTINGS.csv")
print("US_BA_SEARCH_POSTINGS.csv")


ASSIGNMENT 2 - BA POPULATION VERIFICATION
Total job postings: 1,348,454
All explicit BA postings: 4,546
US-search BA postings: 3,732

Expected Assignment 1 counts:
All explicit BA postings: 4,546
US-search BA postings: 3,732

Unique job links:
All BA: 4546
US-search BA: 3732

Saved files:
ALL_BA_POSTINGS.csv
US_BA_SEARCH_POSTINGS.csv


In [4]:

import pandas as pd
import re

# ============================================================
# STEP 2.8: GEOGRAPHIC VALIDATION AND STATE STANDARDISATION
# ============================================================

# Start with the 3,732 US-search Business Analyst postings
us_ba_jobs = us_ba_search.copy()

# ------------------------------------------------------------
# 1. US STATE NAME TO ABBREVIATION MAPPING
# ------------------------------------------------------------

state_mapping = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "District of Columbia": "DC",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY"
}

valid_state_codes = set(state_mapping.values())

# ------------------------------------------------------------
# 2. METROPOLITAN AREA TO STATE MAPPING
# ------------------------------------------------------------

metro_mapping = {
    "Albany, New York Metropolitan Area": "NY",
    "Atlanta Metropolitan Area": "GA",
    "Austin, Texas Metropolitan Area": "TX",
    "Brownsville Metropolitan Area": "TX",
    "Buffalo-Niagara Falls Area": "NY",
    "Charleston, South Carolina Metropolitan Area": "SC",
    "Charlotte Metro": "NC",
    "Cincinnati Metropolitan Area": "OH",
    "Columbia, South Carolina Metropolitan Area": "SC",
    "Columbus, Ohio Metropolitan Area": "OH",
    "Dallas-Fort Worth Metroplex": "TX",
    "Greater Anchorage Area": "AK",
    "Greater Boston": "MA",
    "Greater Cleveland": "OH",
    "Greater Macon": "GA",
    "Greater Minneapolis-St. Paul Area": "MN",
    "Greater Orlando": "FL",
    "Greater Philadelphia": "PA",
    "Greater Phoenix Area": "AZ",
    "Greater Reno Area": "NV",
    "Greater Syracuse-Auburn Area": "NY",
    "Greater Tampa Bay Area": "FL",
    "Greenville-Spartanburg-Anderson, South Carolina Area": "SC",
    "Kansas City Metropolitan Area": "MO",
    "Los Angeles Metropolitan Area": "CA",
    "Miami-Fort Lauderdale Area": "FL",
    "Nashville Metropolitan Area": "TN",
    "New York City Metropolitan Area": "NY",
    "Oklahoma City Metropolitan Area": "OK",
    "Portland, Oregon Metropolitan Area": "OR",
    "San Antonio, Texas Metropolitan Area": "TX",
    "San Francisco Bay Area": "CA",
    "Tallahassee Metropolitan Area": "FL",
    "Utica-Rome Area": "NY",
    "Washington DC-Baltimore Area": "DC"
}

# ------------------------------------------------------------
# 3. EXTRACT AND STANDARDISE US STATES
# ------------------------------------------------------------

def extract_state(location):

    if pd.isna(location):
        return None

    location = str(location).strip()

    # Handle explicitly mapped metropolitan areas
    if location in metro_mapping:
        return metro_mapping[location]

    parts = [part.strip() for part in location.split(",")]

    # Standard format: City, ST
    if len(parts) >= 2:
        if parts[-1].upper() in valid_state_codes:
            return parts[-1].upper()

    # Full state name: California, United States
    if len(parts) >= 2 and parts[-1].lower() in {
        "united states", "usa", "us"
    }:
        state_name = parts[-2]

        if state_name in state_mapping:
            return state_mapping[state_name]

        if state_name.upper() in valid_state_codes:
            return state_name.upper()

    # Full state name only
    if location in state_mapping:
        return state_mapping[location]

    return None


us_ba_jobs["state_clean"] = (
    us_ba_jobs["job_location"].apply(extract_state)
)

# ------------------------------------------------------------
# 4. REPRODUCE ASSIGNMENT 1 GEOGRAPHIC EXCLUSIONS
# ------------------------------------------------------------

# This list reproduces the 10 locations excluded in A1.
# Note: Canada, KY is geographically in Kentucky, USA.
# It is retained here as an A1 methodological replication,
# not because it is a confirmed foreign location.

assignment1_excluded_locations = {
    "Vancouver, British Columbia, Canada",
    "Mexico City Metropolitan Area",
    "Mexico City, Mexico",
    "Tijuana, Baja California, Mexico",
    "Mexicali, Baja California, Mexico",
    "Canada, KY"
}

us_ba_jobs["location_country_issue"] = (
    us_ba_jobs["job_location"]
    .isin(assignment1_excluded_locations)
)

# ------------------------------------------------------------
# 5. CREATE FINAL ASSIGNMENT 1-CONSISTENT US BA DATASET
# ------------------------------------------------------------

us_ba_geo = us_ba_jobs[
    ~us_ba_jobs["location_country_issue"]
].copy()

print("=" * 60)
print("GEOGRAPHIC VALIDATION SUMMARY")
print("=" * 60)

print("Original search-country US records:", len(us_ba_jobs))

print(
    "Country inconsistencies removed:",
    int(us_ba_jobs["location_country_issue"].sum())
)

print("Final US BA geography dataset:", len(us_ba_geo))

print(
    "\nState identified:",
    us_ba_geo["state_clean"].notna().sum()
)

print(
    "State unknown:",
    us_ba_geo["state_clean"].isna().sum()
)

print(
    "State coverage:",
    round(
        us_ba_geo["state_clean"].notna().mean() * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# 6. VERIFY DATASET COUNTS
# ------------------------------------------------------------

assert len(us_ba_jobs) == 3732
assert us_ba_jobs["location_country_issue"].sum() == 10
assert len(us_ba_geo) == 3722
assert us_ba_geo["job_link"].is_unique

print("\nCore geographic checks passed.")

# ------------------------------------------------------------
# 7. SAVE DATASETS
# ------------------------------------------------------------

us_ba_jobs.to_csv(
    OUTPUT_DIR / "US_BA_GEOGRAPHIC_AUDIT.csv",
    index=False
)

us_ba_geo.to_csv(
    OUTPUT_DIR / "US_BA_GEOGRAPHY.csv",
    index=False
)

print("\nFiles saved successfully.")


GEOGRAPHIC VALIDATION SUMMARY
Original search-country US records: 3732
Country inconsistencies removed: 10
Final US BA geography dataset: 3722

State identified: 3708
State unknown: 14
State coverage: 99.62 %

Core geographic checks passed.

Files saved successfully.


In [5]:
print("Skills file:", SKILLS_FILE)
print("Output folder:", OUTPUT_DIR)
print("Validated US BA records:", len(us_ba_geo))

Skills file: E:\Analytic Plan BI\LinkedIn Dataset\job_skills.csv
Output folder: E:\Analytic Plan BI\LinkedIn Dataset\Assignment_2_Analysis
Validated US BA records: 3722


In [6]:

# ============================================================
# STEP 3.1: LOAD AND INTEGRATE US BUSINESS ANALYST SKILLS
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. DEFINE FILE PATHS
# ------------------------------------------------------------

DATA_DIR = Path(r"E:\Analytic Plan BI\LinkedIn Dataset")

SKILLS_FILE = DATA_DIR / "job_skills.csv"

OUTPUT_DIR = DATA_DIR / "Assignment_2_Analysis"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. LOAD VALIDATED US BA DATASET
# ------------------------------------------------------------

us_ba_geo = pd.read_csv(
    OUTPUT_DIR / "US_BA_GEOGRAPHY.csv"
)

print("Validated US BA advertisements:", len(us_ba_geo))

# Verify geographic dataset
assert len(us_ba_geo) == 3722
assert us_ba_geo["job_link"].is_unique

# ------------------------------------------------------------
# 3. EXTRACT UNIQUE BA JOB LINKS
# ------------------------------------------------------------

ba_job_links = set(
    us_ba_geo["job_link"].dropna()
)

print("Unique BA job links:", len(ba_job_links))

# ------------------------------------------------------------
# 4. READ SKILLS DATA IN CHUNKS
# ------------------------------------------------------------

skills_chunks = []

total_skills_rows = 0
chunk_number = 0

for chunk in pd.read_csv(
    SKILLS_FILE,
    usecols=["job_link", "job_skills"],
    chunksize=50000
):

    chunk_number += 1
    total_skills_rows += len(chunk)

    # Retain only skills for validated BA advertisements
    matching_rows = chunk[
        chunk["job_link"].isin(ba_job_links)
    ].copy()

    if not matching_rows.empty:
        skills_chunks.append(matching_rows)

    print(
        f"Chunk {chunk_number}: "
        f"{len(matching_rows)} matching records"
    )

# ------------------------------------------------------------
# 5. COMBINE MATCHING SKILLS RECORDS
# ------------------------------------------------------------

if skills_chunks:
    us_ba_skills_raw = pd.concat(
        skills_chunks,
        ignore_index=True
    )
else:
    us_ba_skills_raw = pd.DataFrame(
        columns=["job_link", "job_skills"]
    )

# ------------------------------------------------------------
# 6. CHECK DATA QUALITY AND JOIN COVERAGE
# ------------------------------------------------------------

matched_jobs = us_ba_skills_raw["job_link"].nunique()

missing_skill_values = (
    us_ba_skills_raw["job_skills"].isna().sum()
)

blank_skill_values = (
    us_ba_skills_raw["job_skills"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

duplicate_skill_links = (
    us_ba_skills_raw["job_link"].duplicated().sum()
)

unmatched_jobs = len(ba_job_links) - matched_jobs

print("\n" + "=" * 60)
print("STEP 3.1: SKILLS INTEGRATION SUMMARY")
print("=" * 60)

print("Total rows scanned:", total_skills_rows)

print("Validated US BA advertisements:", len(us_ba_geo))

print("Matched skills records:", len(us_ba_skills_raw))

print("Unique BA advertisements matched:", matched_jobs)

print("Unmatched BA advertisements:", unmatched_jobs)

print("Missing skill values:", missing_skill_values)

print("Missing or blank skill values:", blank_skill_values)

print("Duplicate skill job links:", duplicate_skill_links)

print(
    "Join coverage:",
    round(matched_jobs / len(us_ba_geo) * 100, 2),
    "%"
)

# ------------------------------------------------------------
# 7. DISPLAY SAMPLE RECORDS
# ------------------------------------------------------------

print("\nSAMPLE SKILLS RECORDS")

display(us_ba_skills_raw.head(10))

# ------------------------------------------------------------
# 8. SAVE EXTRACTED SKILLS DATASET
# ------------------------------------------------------------

output_file = OUTPUT_DIR / "US_BA_SKILLS_RAW.csv"

us_ba_skills_raw.to_csv(
    output_file,
    index=False
)

print("\nSaved successfully:", output_file)


Validated US BA advertisements: 3722
Unique BA job links: 3722
Chunk 1: 23 matching records
Chunk 2: 101 matching records
Chunk 3: 149 matching records
Chunk 4: 547 matching records
Chunk 5: 20 matching records
Chunk 6: 421 matching records
Chunk 7: 26 matching records
Chunk 8: 140 matching records
Chunk 9: 20 matching records
Chunk 10: 33 matching records
Chunk 11: 332 matching records
Chunk 12: 282 matching records
Chunk 13: 111 matching records
Chunk 14: 117 matching records
Chunk 15: 105 matching records
Chunk 16: 98 matching records
Chunk 17: 101 matching records
Chunk 18: 106 matching records
Chunk 19: 119 matching records
Chunk 20: 122 matching records
Chunk 21: 106 matching records
Chunk 22: 110 matching records
Chunk 23: 124 matching records
Chunk 24: 109 matching records
Chunk 25: 89 matching records
Chunk 26: 95 matching records

STEP 3.1: SKILLS INTEGRATION SUMMARY
Total rows scanned: 1296381
Validated US BA advertisements: 3722
Matched skills records: 3606
Unique BA advert

,job_link,job_skills
0,https://www.linkedin.com/jobs/view/business-an...,"SQL Server, Power BI, Business Requirements Ga..."
1,https://www.linkedin.com/jobs/view/senior-it-b...,"Application Configuration, Application Support..."
2,https://www.linkedin.com/jobs/view/calypso-bus...,"investment management, proprietary research, e..."
3,https://www.linkedin.com/jobs/view/calypso-bus...,"Calypso, SQL, Investment Management, Database ..."
4,https://www.linkedin.com/jobs/view/business-an...,"Project management, Organizational/business an..."
5,https://www.linkedin.com/jobs/view/business-an...,"Project Management, Business Analysis, Organiz..."
6,https://www.linkedin.com/jobs/view/procurement...,"Customer Focus, Data Extraction, Data Validati..."
7,https://www.linkedin.com/jobs/view/ui-path-bus...,"UI Path, Business Analyst, Project Manager, SD..."
8,https://www.linkedin.com/jobs/view/is-business...,"SQL Server, Power BI, SQL Reporting, Java, Ris..."
9,https://www.linkedin.com/jobs/view/sr-business...,"Data Analysis, SQL, Tableau, QuickSight, Power..."



Saved successfully: E:\Analytic Plan BI\LinkedIn Dataset\Assignment_2_Analysis\US_BA_SKILLS_RAW.csv


In [7]:

# ============================================================
# STEP 3.2: CLEAN AND STANDARDISE BUSINESS ANALYST SKILLS
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. LOAD EXTRACTED BA SKILLS
# ------------------------------------------------------------

us_ba_skills_raw = pd.read_csv(
    OUTPUT_DIR / "US_BA_SKILLS_RAW.csv"
)

print("Advertisements with skills:",
      us_ba_skills_raw["job_link"].nunique())

# ------------------------------------------------------------
# 2. SPLIT COMMA-SEPARATED SKILLS
# ------------------------------------------------------------

skills_long = us_ba_skills_raw.assign(
    skill=us_ba_skills_raw["job_skills"].str.split(",")
).explode("skill")

print("Skill mentions before cleaning:", len(skills_long))

# ------------------------------------------------------------
# 3. CLEAN SKILL TEXT
# ------------------------------------------------------------

skills_long["skill_clean"] = (
    skills_long["skill"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# Remove missing and empty skills
skills_long = skills_long[
    skills_long["skill_clean"].notna()
    &
    skills_long["skill_clean"].ne("")
].copy()

# ------------------------------------------------------------
# 4. CONSERVATIVE SKILL STANDARDISATION
# ------------------------------------------------------------

# Only merge variants with clearly equivalent meanings.
# Avoid combining distinct concepts without justification.

skill_mapping = {
    "problemsolving": "problem solving"
}

skills_long["skill_standard"] = (
    skills_long["skill_clean"]
    .replace(skill_mapping)
)

# ------------------------------------------------------------
# 5. REMOVE DUPLICATE SKILLS WITHIN EACH ADVERTISEMENT
# ------------------------------------------------------------

skills_standard = skills_long.drop_duplicates(
    subset=["job_link", "skill_standard"]
).copy()

# ------------------------------------------------------------
# 6. SUMMARISE SKILL FREQUENCY
# ------------------------------------------------------------

skills_denominator = (
    us_ba_skills_raw["job_link"].nunique()
)

skill_summary = (
    skills_standard
    .groupby("skill_standard")["job_link"]
    .nunique()
    .reset_index(name="Postings")
    .sort_values("Postings", ascending=False)
)

skill_summary["Percentage"] = (
    skill_summary["Postings"]
    / skills_denominator
    * 100
).round(2)

# ------------------------------------------------------------
# 7. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 3.2: SKILLS CLEANING SUMMARY")
print("=" * 60)

print("Skill mentions after cleaning:", len(skills_long))
print("Unique job-skill pairs:", len(skills_standard))
print("Unique standardised skills:",
      skills_standard["skill_standard"].nunique())
print("Skills prevalence denominator:", skills_denominator)

assert skills_denominator == 3606

assert not skills_standard.duplicated(
    subset=["job_link", "skill_standard"]
).any()

assert skill_summary["Percentage"].between(0, 100).all()

print("\nValidation checks passed.")

# ------------------------------------------------------------
# 8. DISPLAY TOP 30 SKILLS
# ------------------------------------------------------------

print("\nTOP 30 BUSINESS ANALYST SKILLS")

display(skill_summary.head(30))

# ------------------------------------------------------------
# 9. SAVE CLEANED DATASETS
# ------------------------------------------------------------

skills_standard.to_csv(
    OUTPUT_DIR / "US_BA_SKILLS_STANDARDISED.csv",
    index=False
)

skill_summary.to_csv(
    OUTPUT_DIR / "US_BA_SKILL_SUMMARY.csv",
    index=False
)

print("\nFiles saved successfully.")


Advertisements with skills: 3606
Skill mentions before cleaning: 92649

STEP 3.2: SKILLS CLEANING SUMMARY
Skill mentions after cleaning: 92649
Unique job-skill pairs: 92020
Unique standardised skills: 22189
Skills prevalence denominator: 3606

Validation checks passed.

TOP 30 BUSINESS ANALYST SKILLS


,skill_standard,Postings,Percentage
2952,business analysis,1475,40.90
15715,project management,1232,34.17
5712,data analysis,1214,33.67
4484,communication,1069,29.65
15044,problem solving,974,27.01
18644,sql,916,25.40
1562,analytical skills,750,20.80
4531,communication skills,591,16.39
16744,requirements gathering,576,15.97
19863,teamwork,574,15.92



Files saved successfully.


In [8]:

# ============================================================
# STEP 3.3: DATA PREPARATION QUALITY SUMMARY
# ============================================================

import pandas as pd

# Load final datasets to make this cell reproducible
us_ba_geo = pd.read_csv(
    OUTPUT_DIR / "US_BA_GEOGRAPHY.csv"
)

us_ba_skills_raw = pd.read_csv(
    OUTPUT_DIR / "US_BA_SKILLS_RAW.csv"
)

skills_standard = pd.read_csv(
    OUTPUT_DIR / "US_BA_SKILLS_STANDARDISED.csv"
)

# Core counts
n_ba = us_ba_geo["job_link"].nunique()
n_skills = us_ba_skills_raw["job_link"].nunique()

n_mentions = len(
    us_ba_skills_raw["job_skills"]
    .str.split(",")
    .explode()
)

n_pairs = len(skills_standard)

quality_summary = pd.DataFrame({
    "Metric": [
        "Validated US BA advertisements",
        "Advertisements with matched skills",
        "Advertisements without matched skills",
        "Skills join coverage (%)",
        "Skill mentions before deduplication",
        "Unique job-skill pairs",
        "Duplicate skill mentions removed",
        "Distinct standardised skill labels",
        "Duplicate job-skill pairs remaining",
        "State identification coverage (%)"
    ],
    "Value": [
        n_ba,
        n_skills,
        n_ba - n_skills,
        round(n_skills / n_ba * 100, 2),
        n_mentions,
        n_pairs,
        n_mentions - n_pairs,
        skills_standard["skill_standard"].nunique(),
        skills_standard.duplicated(
            ["job_link", "skill_standard"]
        ).sum(),
        round(
            us_ba_geo["state_clean"].notna().mean() * 100,
            2
        )
    ]
})

print("DATA PREPARATION QUALITY SUMMARY")
print("=" * 60)

display(quality_summary)

quality_summary.to_csv(
    OUTPUT_DIR / "DATA_QUALITY_SUMMARY.csv",
    index=False
)

print("\nData quality summary saved successfully.")


DATA PREPARATION QUALITY SUMMARY


,Metric,Value
0,Validated US BA advertisements,3722.00
1,Advertisements with matched skills,3606.00
2,Advertisements without matched skills,116.00
3,Skills join coverage (%),96.88
4,Skill mentions before deduplication,92649.00
5,Unique job-skill pairs,92020.00
6,Duplicate skill mentions removed,629.00
7,Distinct standardised skill labels,22189.00
8,Duplicate job-skill pairs remaining,0.00
9,State identification coverage (%),99.62



Data quality summary saved successfully.
